# Combat population: random reward preferences and a shared test

Train independent PPO pilots against BVR Sim's **simple opponent**, using JSBSim.
Defaults: **10 pilots, 150 epochs, 75 fresh training scenarios per epoch**,
50 fixed validation scenarios and 100 separate final test scenarios per pilot.
The transformer, actor and existing critic retain the same architecture as notebook 04.
Pilots train sequentially; CPU processes collect flights concurrently and CUDA graphs
accelerate GPU updates. Each pilot gets its own weights, optimizer and checkpoint.

Training rewards use the original combat event formula with coefficients drawn uniformly
from explicit ranges. There is no Halton sequence or coefficient-budget normalization.
Different preferences are experiments; flying styles must be measured after training.

**Comparison priority: eliminate the opponent and survive.** Every selected checkpoint
faces the same fresh test cases. Report surviving wins, mutual destruction, survival,
eliminations and behaviour measurements together. Own-objective validation return selects
each pilot's checkpoint; the test never selects checkpoints or updates the policy.

The [previous published results](results/20261006_notebook05.ipynb) remain available with
their tables and embedded figures. They describe the earlier recipe, not this revision.
New runs use `artifacts/rl_pilot_notebook/combat/<timestamp>/` and never overwrite old runs.
Notebook 04 retains its original defaults; [notebook 06](06_train_multiple_pilots.ipynb)
retains its separate hybrid evade/pursue/eliminate experiment.

## Setup

Install dependencies from the repository root and restart the kernel:

```bash
python -m pip install -e '.[ml,video]'
```

Select the project environment in VS Code. CUDA-enabled PyTorch is required for GPU
inference and updates; simulation and opponent control run in persistent CPU processes.
Workers default to 80% of logical CPUs, rounded down, with a minimum of one. PyTorch
CPU threads default to at most eight. These limits do not reserve cores or guarantee
a CPU percentage. `BVR_PILOT_WORKERS`, `BVR_PILOT_TORCH_THREADS`, and `BVR_PILOT_DEVICE`
override them. Keep `simulator_executor: process` for parallel CPU simulation.

Restart the kernel and run from the top after a code update. For a short installation
check, set epochs, episode seconds and PPO passes to 1, and set **training, validation
and test scenario counts to 3** (`BVR_PILOT_SCENARIOS`,
`BVR_PILOT_EVALUATION_SCENARIOS`, `BVR_PILOT_TEST_SCENARIOS`). Set `BVR_PILOT_COUNT=1`
for a single-pilot smoke run. All environment overrides are read at setup/configuration.

CPU load naturally falls during GPU updates. Saved phase timings distinguish collection,
updates, validation and replay writing. PPO now uses at most 10 passes and minibatches of
256 decisions, stopping after a pass when approximate KL exceeds 1.5 times the 0.02 target.
This is a stability safeguard, not a guarantee of better performance or a hard KL bound.
CUDA graphs remain enabled; `BVR_PILOT_CUDA_GRAPHS=0` disables them for comparison.
The default learning rate is `0.00003`, following the completed controlled experiment. Set `BVR_PILOT_LEARNING_RATE` to override it. The selected-skill probability experiment is launched separately from the [experiment index](../docs/pilot_experiments.md); the training notebook retains the original probability calculation.


In [ ]:
import json
import os
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Leave CPU capacity for the editor and other applications.
DEFAULT_SIMULATOR_WORKERS = max(1, ((os.cpu_count() or 1) * 80) // 100)
TORCH_THREADS = int(os.getenv(
    "BVR_PILOT_TORCH_THREADS", str(min(8, DEFAULT_SIMULATOR_WORKERS))
))
if TORCH_THREADS < 1:
    raise ValueError("BVR_PILOT_TORCH_THREADS must be positive")
torch.set_num_threads(TORCH_THREADS)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "bvr_behavior_prediction").exists():
    REPO_ROOT = REPO_ROOT.parent
for source_root in (REPO_ROOT, REPO_ROOT / "bvr_sim_source"):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from bvr_behavior_prediction.rl.bvr_jsbsim_backend import BVRJSBSimBackend
from bvr_behavior_prediction.rl.config import PilotTrainingConfig, RewardWeights
from bvr_behavior_prediction.rl.environment import BluePilotEnvironment
from bvr_behavior_prediction.rl.multi_pilot import PilotSpec, train_pilots, load_trained_pilot
from bvr_behavior_prediction.rl.reward import combat_reward_definition
from bvr_behavior_prediction.rl.reward_population import (
    DEFAULT_COMBAT_REWARD_RANGES, random_combat_population_weights,
)
from bvr_behavior_prediction.rl.trainer import PPOTrainer

SEED = int(os.getenv("BVR_PILOT_SEED", "7"))
# A separate random stream controls reward sampling, not network/scenario seeds.
REWARD_SEED = int(os.getenv("BVR_PILOT_REWARD_SEED", "42"))
# Matched initialization and scenarios isolate reward preferences.
SEED_STRIDE = int(os.getenv("BVR_PILOT_SEED_STRIDE", "0"))
evaluation_seed_value = os.getenv("BVR_PILOT_EVALUATION_SEED")
EVALUATION_SEED = int(evaluation_seed_value) if evaluation_seed_value else None
EPOCHS = int(os.getenv("BVR_PILOT_EPOCHS", "150"))
EPISODE_SECONDS = float(os.getenv("BVR_PILOT_EPISODE_SECONDS", "120"))
PILOT_COUNT = int(os.getenv("BVR_PILOT_COUNT", "10"))
if PILOT_COUNT < 1:
    raise ValueError("BVR_PILOT_COUNT must be positive")
BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = Path(os.getenv(
    "BVR_PILOT_OUTPUT", REPO_ROOT / "artifacts/rl_pilot_notebook/combat" / BATCH_ID
)).resolve()
DEVICE = os.getenv("BVR_PILOT_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print({"device": DEVICE, "pilot_count": PILOT_COUNT, "epochs_per_pilot": EPOCHS,
       "torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "torch_threads": torch.get_num_threads(), "output": str(OUTPUT_DIR)})

## BVR Sim backend with JSBSim flight dynamics

Each independent pilot controls one F-16 through the complete `SkillManager` catalogue.
The opponent uses BVR Sim's existing simple baseline. The adapter translates tactical
skills into flight controls and records ACMI telemetry. Logs are isolated by pilot.

In [ ]:
BASE_OBSERVATION_SIZE = BVRJSBSimBackend.OBSERVATION_SIZE
ENERGY_FEATURES = 4
OBSERVATION_SIZE = BASE_OBSERVATION_SIZE + ENERGY_FEATURES


def make_environment_factory(run_config):
    def bvr_jsbsim_backend_factory(scenario_dict, recording_path=None):
        return BVRJSBSimBackend(
            scenario_dict,
            recording_path=recording_path,
            log_dir=run_config.output_dir / "simulator_logs",
        )

    def make_environment(scenario, recording_path=None):
        return BluePilotEnvironment(
            bvr_jsbsim_backend_factory,
            scenario,
            planning_horizon_s=run_config.planning_horizon_s,
            episode_duration_s=run_config.episode_duration_s,
            history_duration_s=run_config.history_duration_s,
            sample_interval_s=run_config.sample_interval_s,
            recording_path=recording_path,
        )

    return make_environment

## Sample reward preferences within explicit ranges

Every pilot samples its actual coefficients uniformly within `REWARD_RANGES`. A fixed
`REWARD_SEED` reproduces the roster; changing it draws another roster. Extending the
pilot count preserves existing entries. The network/scenario seed is separate. No pilot
is automatically assigned the original weights, and no final normalization changes the
displayed ranges. Equal bounds fix a coefficient.

| Event | Default coefficient range | Purpose |
| --- | --- | --- |
| Opponent destruction | +250 fixed | Shared training unit for success |
| Crash | -450 to -300 | Make aircraft loss costly |
| Shot down | -450 to -300 | Vary tolerance for risk |
| Missile avoided | 0 to +15 | Vary emphasis on surviving missile threats |
| Lock acquired | 0 to +2 | Small guidance towards weapon employment |
| Locked launch | 0 to +2 | Keep launching subordinate to combat outcomes |
| Unlocked launch | -12 to -4 | Vary penalties for poor weapon use |
| Safe airborne decision | +0.02 fixed | Retain the original small survival term |

These are editable hypotheses, not calibrated optimal values. Destruction plus an
aircraft-loss penalty is negative for every default pilot. Intermediate bonuses can
still offset that loss; training reward is therefore not the final outcome score.
Event repetition rules remain those of `CombatReward`, including repeated locked
launch rewards. The lower launch range reduces that incentive but does not eliminate it.
Opponent destruction includes any cause; the comparison's clean win requires a
**confirmed missile elimination and survival**.

The same training seed gives pilots matched initialization and a matched sequence of
fresh scenario batches. Validation remains fixed. Each pilot selects its checkpoint
using its own objective to retain its preferences. A final, separate test compares
outcomes and behaviour without using those individual reward coefficients.

The original common combat reward is retained as a secondary diagnostic only. It is
not the primary ranking because launch/evasion bonuses can change its ranking.

In [ ]:
REWARD_RANGES = dict(DEFAULT_COMBAT_REWARD_RANGES)
# Edit actual coefficient intervals, for example:
# REWARD_RANGES["incoming_missile_avoided"] = (0.0, 20.0)
REWARD_WEIGHTS = random_combat_population_weights(
    PILOT_COUNT, seed=REWARD_SEED, ranges=REWARD_RANGES,
)


def make_reward_definition(pilot_index):
    return combat_reward_definition(
        f"combat_objective_{pilot_index + 1:03d}", weights=REWARD_WEIGHTS[pilot_index],
    )


PILOTS = [
    PilotSpec(
        pilot_id=f"pilot_{index + 1:03d}",
        seed=SEED + SEED_STRIDE * index,
        reward=make_reward_definition(index),
    )
    for index in range(PILOT_COUNT)
]
POPULATION_METADATA = {
    "reward_sampler": "independent_uniform_v1",
    "reward_seed": REWARD_SEED,
    "reward_ranges": REWARD_RANGES,
    "comparison_priority": "confirmed elimination while surviving",
}
COMMON_REWARD = combat_reward_definition("common_combat", weights=RewardWeights())
display(pd.DataFrame([
    {"pilot_id": pilot.pilot_id, "seed": pilot.seed,
     **pilot.reward.parameters["weights"]}
    for pilot in PILOTS
]))

In [ ]:
config = PilotTrainingConfig(
    seed=SEED,
    evaluation_seed=EVALUATION_SEED,
    epochs=EPOCHS,
    episode_duration_s=EPISODE_SECONDS,
    planning_horizon_s=1.0,
    history_duration_s=2.0,
    sample_interval_s=0.1,
    transformer_heads=4,
    transformer_layers=2,
    scenarios_per_epoch=int(os.getenv("BVR_PILOT_SCENARIOS", "75")),
    evaluation_scenarios_per_epoch=int(os.getenv("BVR_PILOT_EVALUATION_SCENARIOS", "50")),
    simulator_workers=int(os.getenv("BVR_PILOT_WORKERS", str(DEFAULT_SIMULATOR_WORKERS))),
    simulator_executor=os.getenv("BVR_PILOT_EXECUTOR", "process"),
    hidden_size=64,
    learning_rate=float(os.getenv("BVR_PILOT_LEARNING_RATE", "0.00003")),
    update_epochs=int(os.getenv("BVR_PILOT_UPDATE_EPOCHS", "10")),
    minibatch_size=int(os.getenv("BVR_PILOT_MINIBATCH", "256")),
    cuda_graph_updates=os.getenv("BVR_PILOT_CUDA_GRAPHS", "1") != "0",
    resample_training_scenarios=os.getenv("BVR_PILOT_RESAMPLE", "1") != "0",
    training_reward_scale=float(os.getenv("BVR_PILOT_REWARD_SCALE", "0.01")),
    target_kl=float(os.getenv("BVR_PILOT_TARGET_KL", "0.02")),
    log_ppo_diagnostics=True,
    test_scenarios_per_pilot=int(os.getenv("BVR_PILOT_TEST_SCENARIOS", "100")),
    test_seed=int(os.environ["BVR_PILOT_TEST_SEED"]) if os.getenv("BVR_PILOT_TEST_SEED") else None,
    checkpoint_interval=max(1, EPOCHS // 5),
    diagnostic_interval=1,
    output_dir=OUTPUT_DIR,
    mlflow_experiment="jsbsim-combat-population",
    mlflow_tracking_uri=f"sqlite:///{(REPO_ROOT / 'artifacts/mlflow.db').as_posix()}",
)
# Explicit pilot definitions above override the legacy config.reward field.
print(json.dumps({key: value for key, value in config.as_dict().items()
                  if key != "reward"}, indent=2))

## Train, select and test each pilot

This cell trains fresh models. Each epoch collects a new matched batch of training
conditions, computes GAE and runs up to 10 PPO passes with KL stopping. A fixed factor
of 0.01 scales rewards **only for learning**: the critic and GAE use those smaller units,
while all reported returns/components keep their original points. This changes the
balance of critic and actor gradients without changing the relative reward preferences.

Validation runs on the first, every fifth and final epoch and selects `best_model.pt`
under the pilot's own reward. That selected model is then evaluated on the common
validation reward and once on the **same fresh final test set** for all pilots. Test
seeds are disjoint from every training epoch and validation; the complete test cases
are sampled as one batch and reused across pilots. Test results never select a model.

The batch saves reward ranges/seed, per-epoch scenario seed ranges, checkpoints, logs,
and individual test outcomes. It refuses an existing output directory. Rerunning setup
creates a new timestamp. More pilots multiply the training work.

`selection_return` uses individual preferences and is not a cross-pilot ranking.
The test table's primary measure is `clean_win_rate`. Its shared secondary outcome score
is **+100 for survival with a confirmed elimination, 0 for survival without one,
-50 for mutual destruction, and -100 for aircraft loss with the opponent still alive**.
This score expresses an explicit preference; it is not a neutral measure of optimality.

In [ ]:
results = train_pilots(
    pilots=PILOTS,
    config=config,
    env_factory_builder=make_environment_factory,
    observation_size=OBSERVATION_SIZE,
    device=DEVICE,
    benchmark_reward=COMMON_REWARD,
    population_metadata=POPULATION_METADATA,
)
summary = pd.DataFrame(results)
display(summary[["pilot_id", "selection_return", "benchmark_return", "checkpoint"]])
test_summary = pd.DataFrame([
    {"pilot_id": result["pilot_id"], **result["test"]}
    for result in results if "test" in result
])
if not test_summary.empty:
    display(test_summary[[
        "pilot_id", "clean_win_rate", "clean_win_ci95", "mutual_destruction_rate",
        "survival_rate", "elimination_rate", "outcome_score",
    ]].sort_values(["clean_win_rate", "outcome_score"], ascending=False))
print(f"Batch manifest: {OUTPUT_DIR / 'pilots.json'}")

In [ ]:
# Select any completed pilot for inspection. This does not rerun training.
SELECTED_PILOT = os.getenv("BVR_PILOT_VIEW", results[0]["pilot_id"])
selected_result = next(result for result in results if result["pilot_id"] == SELECTED_PILOT)
selected_spec = next(pilot for pilot in PILOTS if pilot.pilot_id == SELECTED_PILOT)
RUN_DIR = OUTPUT_DIR / selected_result["output_dir"]
selected_config = replace(
    config, seed=selected_spec.seed,
    evaluation_seed=selected_result["evaluation_seed"], output_dir=RUN_DIR,
)
trainer = PPOTrainer(
    make_environment_factory(selected_config), OBSERVATION_SIZE, selected_config,
    device=DEVICE, reward_definition=selected_spec.reward,
)
trainer.pilot.load_state_dict(load_trained_pilot(RUN_DIR, DEVICE).state_dict())
trained_pilot = trainer.pilot.eval()

FINAL_EVALUATION_DIR = RUN_DIR / "final_evaluation"
FINAL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
final_evaluation_rollouts = []
final_evaluation_manifest = []
for index, scenario in enumerate(trainer.evaluation_scenarios):
    episode_seed = selected_config.resolved_evaluation_seed + index
    acmi_path = FINAL_EVALUATION_DIR / f"scenario_{index:03d}_seed_{episode_seed}.txt.acmi"
    rollout, score, final_info = trainer._episode(
        scenario, episode_seed, recording_path=acmi_path, deterministic=True,
    )
    final_evaluation_rollouts.append(rollout)
    final_evaluation_manifest.append({
        "pilot_id": SELECTED_PILOT,
        "checkpoint_sha256": selected_result["checkpoint_sha256"],
        "reward": selected_spec.reward.as_dict(),
        "scenario_index": index, "seed": episode_seed, "score": score,
        "scenario": scenario.as_dict(), "final_info": final_info, "acmi": acmi_path.name,
    })

manifest_path = FINAL_EVALUATION_DIR / "manifest.json"
manifest_path.write_text(json.dumps(final_evaluation_manifest, indent=2, default=str), encoding="utf8")
print(f"Recorded {len(final_evaluation_manifest)} replays for {SELECTED_PILOT}")
print(f"ACMI directory: {FINAL_EVALUATION_DIR}")

## Compare competence and observed behaviour separately

The **fresh test** table compares all pilots on identical conditions. Prioritise surviving
wins, read the approximate 95% binomial interval, and inspect mutual destruction,
survival and eliminations. Intervals describe variation across scenarios, not variation
between repeated training runs. Overlapping intervals do not establish an ordering.
The four outcome categories partition all flights, including time-limit survivors.

Missile expenditure, missile avoidance, threat exposure and skill frequencies describe
observed behaviour. They need not improve together. Avoidance depends on how many
threats were encountered; skill frequency alone is not evidence of skill effectiveness.
Per-flight `test_episodes.json` records allow paired comparisons by scenario and seed.

The validation component table remains available as a secondary diagnostic. It contains
**mean cumulative points per flight under the original common reward**, not the sampled
coefficients. Its rows sum to the original common validation return. Unlike the fresh
test, validation selected checkpoints, so it is not an untouched assessment.

Training/validation return curves use each pilot's own reward. Policy/value loss,
entropy, approximate KL, clipping fraction and critic explained variance are now logged.
Loss is a learning diagnostic, not a combat score. More varied rewards do not guarantee
distinct styles, and these changes need a new run to establish performance.

In [ ]:
batch_manifest = json.loads((OUTPUT_DIR / "pilots.json").read_text(encoding="utf8"))
completed = [pilot for pilot in batch_manifest["pilots"] if pilot["status"] == "completed"]
summary = pd.DataFrame(completed)
metrics_by_pilot = {
    pilot["pilot_id"]: pd.read_json(
        OUTPUT_DIR / pilot["output_dir"] / "training_metrics.jsonl", lines=True
    )
    for pilot in completed
}
test_summary = pd.DataFrame([
    {"pilot_id": pilot["pilot_id"], **pilot["test"]}
    for pilot in completed if "test" in pilot
])
if not test_summary.empty:
    print("Fresh test: competence (same cases and scoring for every pilot)")
    display(test_summary[[
        "pilot_id", "scenarios", "clean_win_rate", "clean_win_ci95",
        "mutual_destruction_rate", "survival_rate", "elimination_rate", "outcome_score",
    ]].sort_values(["clean_win_rate", "outcome_score"], ascending=False))
    print("Fresh test: observed behaviour")
    display(test_summary[[
        "pilot_id", "mean_missiles_launched", "mean_missiles_avoided",
        "mean_threatened_seconds", "mean_duration_seconds",
    ]])
    display(pd.DataFrame({
        pilot["pilot_id"]: pilot["test"]["skill_decision_fractions"]
        for pilot in completed if "test" in pilot
    }).fillna(0.0).T.rename_axis("pilot_id"))
else:
    print("No fresh test recorded for this batch; validation is not a held-out test.")

print("Secondary diagnostic: common validation reward components (mean points per flight)")
display(pd.DataFrame([
    {"pilot_id": pilot["pilot_id"], **pilot["benchmark_component_means"]}
    for pilot in completed
]).set_index("pilot_id"))

In [ ]:
figure, axes = plt.subplots(len(completed), 2, figsize=(12, 4 * len(completed)), squeeze=False)
for row, pilot in enumerate(completed):
    metrics = metrics_by_pilot[pilot["pilot_id"]]
    evaluated = metrics[metrics["evaluation_performed"] == 1]
    axes[row, 0].plot(evaluated["epoch"], evaluated["mean_return"], marker="o", label="Validation")
    axes[row, 0].plot(metrics["epoch"], metrics["training_mean_return"], alpha=0.4, label="Training")
    axes[row, 0].plot(metrics["epoch"], metrics["best_return"], linestyle="--", label="Best")
    axes[row, 0].set(title=f"{pilot['pilot_id']}: own reward", xlabel="Epoch", ylabel="Return")
    axes[row, 0].legend()
    if {"policy_loss", "value_loss"}.issubset(metrics.columns):
        axes[row, 1].plot(metrics["epoch"], metrics["policy_loss"], label="Policy")
        axes[row, 1].plot(metrics["epoch"], metrics["value_loss"], label="Value (learning units)")
        axes[row, 1].legend()
    else:
        axes[row, 1].plot(metrics["epoch"], metrics["loss"], color="tab:orange")
    axes[row, 1].set(title=f"{pilot['pilot_id']}: PPO diagnostics", xlabel="Epoch", ylabel="Loss")
    for axis in axes[row]:
        axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

if not test_summary.empty:
    rate = test_summary["clean_win_rate"].to_numpy()
    bounds = np.asarray(test_summary["clean_win_ci95"].tolist())
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.bar(test_summary["pilot_id"], rate,
             yerr=np.maximum(0, np.vstack((rate - bounds[:, 0], bounds[:, 1] - rate))), capsize=4)
    axis.set(title="Fresh test: confirmed elimination while surviving",
             xlabel="Pilot", ylabel="Clean win rate (95% Wilson interval)", ylim=(0, 1))
    axis.tick_params(axis="x", labelrotation=30)
    figure.tight_layout()
    plt.show()

    behaviour = test_summary.dropna(subset=["mean_missiles_launched", "mean_threatened_seconds"])
    if not behaviour.empty:
        figure, axis = plt.subplots(figsize=(9, 5))
        points = axis.scatter(behaviour["mean_missiles_launched"], behaviour["mean_threatened_seconds"],
                              c=behaviour["clean_win_rate"], vmin=0, vmax=1, s=90, cmap="viridis")
        for _, pilot in behaviour.iterrows():
            axis.annotate(pilot["pilot_id"],
                          (pilot["mean_missiles_launched"], pilot["mean_threatened_seconds"]),
                          xytext=(5, 5), textcoords="offset points")
        axis.set(title="Observed behaviour on the same test cases",
                 xlabel="Mean missiles launched", ylabel="Mean seconds under missile threat")
        figure.colorbar(points, ax=axis, label="Clean win rate")
        figure.tight_layout()
        plt.show()

## Inspect the selected pilot

The replay cell saves the selected pilot's checkpoint hash and reward definition alongside
every ACMI flight. Change `BVR_PILOT_VIEW` or `SELECTED_PILOT` in that cell to inspect
another completed pilot. Open the `.txt.acmi` files in Tacview for the full flights.

In [ ]:
rollout = final_evaluation_rollouts[0]
first_evaluation = final_evaluation_manifest[0]
evaluation_return = first_evaluation["score"]
final_info = first_evaluation["final_info"]
print(f"Example ACMI: {FINAL_EVALUATION_DIR / first_evaluation['acmi']}")
selected_skills = [trained_pilot.skill_names[index] for index in rollout.skills]
skill_counts = pd.Series(selected_skills).value_counts().rename_axis("skill").rename("decisions")
print(f"Evaluation return: {evaluation_return:.2f}")
display(skill_counts.to_frame())

skill_counts.sort_values().plot.barh(figsize=(8, max(3, len(skill_counts) * 0.35)))
plt.title("Skills selected in the first fixed evaluation rollout")
plt.xlabel("Decisions")
plt.tight_layout()
plt.show()


## Next experiments

- Run this recipe and compare outcomes on the same fresh cases, then inspect replays
  before assigning style labels. Different rewards can still learn similar behaviour.
- Repeat the roster with several training seeds. A scenario confidence interval does
  not measure sensitivity to neural-network initialization or action sampling.
- Freeze a new final test suite before another round of tuning. Repeatedly choosing
  experiments based on this test eventually makes it another validation set.
- Evaluate the earlier published checkpoints on that same test suite for a fair
  before/after comparison. Their old validation figures are not directly comparable
  with the new test results.
- Inspect `approx_kl`, `clip_fraction`, `explained_variance`, `value_loss`,
  `policy_loss`, `update_passes_completed` and phase timings before increasing epochs.
- Keep the baseline opponent fixed for this experiment; later test other opponents
  or an opponent population to measure generalization.
- Saved checkpoints contain inference weights, not optimizer state for exact resume.